# Reproduce the manuscript analysis

This is the canonical, restart-and-run notebook for the released numerical record. It **does not call hosted models or download the benchmark corpora**. Instead, it reconstructs every manuscript table and figure from the persisted CSV/JSON outputs, reruns the corrected closed-label scoring, and checks the central counts and effects.

The notebook works in Google Colab after uploading or cloning this repository. No credentials are required.

In [ ]:
# Install the analysis-only dependencies in Google Colab.
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "numpy>=1.26", "pandas>=2.0", "scipy>=1.11",
                       "scikit-learn>=1.4", "matplotlib>=3.8", "pytest>=8.0"])
print("Dependencies installed")

In [ ]:
# Locate the repository root.
from pathlib import Path
import os

def is_repo(path: Path) -> bool:
    return (path / "scripts" / "recompute_manuscript_results.py").is_file()

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
# In Colab, a cloned repository is usually an immediate child of /content.
if Path("/content").is_dir():
    candidates.extend(Path("/content").iterdir())
# Also inspect immediate child directories of the current working directory.
candidates.extend(p for p in cwd.iterdir() if p.is_dir())
REPO = next((p for p in candidates if is_repo(p)), None)
if REPO is None:
    raise FileNotFoundError(
        "Repository root not found. Clone/upload the repository, then set REPO "
        "to the directory containing scripts/recompute_manuscript_results.py."
    )
os.chdir(REPO)
print("Repository:", REPO.resolve())

In [ ]:
# Recompute all derived CSVs, LaTeX fragments, figures and verification summaries.
import subprocess, sys
subprocess.check_call([sys.executable, "scripts/recompute_manuscript_results.py"])
subprocess.check_call([sys.executable, "scripts/verify_manuscript_claims.py"])
subprocess.check_call([sys.executable, "scripts/audit_manuscript_tables.py"])
print("Analysis, claim verification and independent table audit completed")

In [ ]:
# Run release integrity tests after the independent audit has been generated.
subprocess.check_call([sys.executable, "-m", "pytest", "-q", "tests/test_release.py"])

In [ ]:
# Inspect the central verification record.
import json
from pprint import pprint
summary = json.loads(Path("results/derived/verification_summary.json").read_text())
pprint(summary)

In [ ]:
# Core Axis-1 ordering.
import pandas as pd
rank = pd.read_csv("results/derived/axis1_rank_inversion.csv")
rank

In [ ]:
# Factorial mechanism experiment.
contrasts = pd.read_csv("results/derived/size_balance_contrasts.csv")
contrasts

In [ ]:
# Corrected closed-label Axis-2 result.
axis2 = pd.read_csv("results/derived/axis2_closed_label_summary.csv")
axis2[["model", "accuracy", "macro_f1_closed", "parse_rate", "mean_delta",
       "p_bonferroni_6", "usd"]]

## Interpretation guardrails

- TabSTAR is not included in the strict Axis-1 rank panel or the 48 paired comparisons because its CIC run uses a different held-out partition and a larger per-class fitting budget.
- The large classical arm is capped at 400,000 fitting records; it is not called “full data” in the manuscript.
- The 2x2 experiment separates fitting-set size from class distribution; its result supports class balance as the principal measured lever.
- Axis-2 parse failures are errors for the true label, while macro-F1 is averaged only over the declared closed label set.
- Non-significance is not interpreted as equivalence.

## Independent statistical audit

In [ ]:
audit = json.loads(Path("results/derived/table_statistical_audit.json").read_text())
print(audit["status"], "-", len(audit["checks"]), "checks")
[a["name"] for a in audit["checks"]]